# 上下文工程——使用 Session 管理短期记忆

AI 智能体常常处于**长时间运行、多轮次的交互**中，在这种情况下，保持**上下文**的恰当平衡至关重要。如果携带过多内容，模型可能会分心、效率低下，甚至彻底失败；如果保留过少，智能体就会失去连贯性。 

这里的上下文指模型一次能够关注的全部 token 窗口（输入 + 输出）。对于 [GPT-5](https://platform.openai.com/docs/models/gpt-5)，这一容量最多可达 272k 输入 token 和 128k 输出 token，但即便是如此大的窗口，也可能被未经整理的历史记录、冗余的工具结果或嘈杂的检索内容所淹没。这使得上下文管理不仅是一种优化，更是一种必需。

在本 cookbook 中，我们将探讨如何**使用 [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) 中的 `Session` 对象有效管理上下文**，重点介绍两种经过验证的上下文管理技术——**裁剪（trimming）**和**压缩（compression）**——以保持智能体快速、可靠且具有成本效益。

#### 为什么上下文管理很重要

* **在长对话中保持连贯** – 让智能体始终锚定在用户的最新目标上，而不是拖着过时的细节。会话级裁剪和摘要可以防止“昨天的计划”覆盖今天的请求。
* **更高的工具调用准确率** – 聚焦的上下文能改善函数选择和参数填充，在多工具运行中减少重试、超时和连锁失败。
* **更低的延迟和成本** – 更小、更精炼的提示可以减少每轮的 token 数和注意力负担。
* **遏制错误和幻觉** – 摘要就像“洁净室”，可以纠正或省略先前的错误；裁剪则可以避免错误事实被一轮又一轮地放大（“上下文中毒”）。
* **更易于调试和观测** – 稳定的摘要和有界的历史记录让日志更具可比性：你可以对比摘要差异、定位退化原因，并可靠地复现故障。
* **应对多问题和交接的韧性** – 在涉及多个问题的对话中，针对每个问题的小型摘要可以让智能体暂停/恢复、升级给人工处理或移交给另一个智能体，同时保持一致。


![AI 智能体中的记忆对比](../../images/memory_comparison.jpg)

[OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses/create#responses-create-previous_response_id) 通过内置状态以及使用 `previous_response_id` 的消息链，提供了**基础的记忆支持**。

你可以将上一个响应的 `id` 作为 `previous_response_id` 传入来延续对话，也可以手动管理上下文：把输出收集到列表中，再将其作为下一个响应的 `input` 重新提交。

你无法获得的是**自动记忆管理**。这正是 **Agents SDK** 的用武之地。它在 Responses 之上提供了[会话记忆](https://openai.github.io/openai-agents-python/sessions/)，因此你不再需要手动追加 `response.output` 或自行跟踪 ID。会话本身就成为**记忆对象**：你反复调用 `Runner.run(agent, "...", session=session)`，SDK 会负责处理上下文长度、历史记录和连续性——让构建连贯的多轮智能体变得容易得多。


#### 真实场景

我们将通过一个常见的长时间运行任务的实际示例来讲解这些技术，例如：

* **多轮客户服务对话**
在关于科技产品（涵盖硬件和软件）的长时间对话中，客户往往会随着时间推移提出多个问题。智能体必须保持一致并聚焦于目标，只保留要点，而不是拖着过去的每一个细节。

#### 涵盖的技术

为应对这些挑战，我们将介绍使用 OpenAI Agents SDK 的两种独立的具体方法：

- **上下文裁剪** – 丢弃较早的轮次，同时保留最近的 N 轮。
  - **优点**

    * **确定且简单：** 没有摘要器带来的不确定性；状态易于推理，运行易于复现。
    * **零额外延迟：** 无需额外调用模型来压缩历史记录。
    * **保真保留最近的工作：** 最新的工具结果、参数和边界情况都按原样保留——非常适合调试。
    * **“摘要漂移”风险更低：** 你从不重新解读或压缩事实。

    **缺点**

    * **突然遗忘长程上下文：** 重要的早期约束、ID 或决策一旦滚出最近 N 轮范围就可能消失。
    * **用户体验上的“失忆”：** 在长会话进行到一半时，智能体可能看起来“忘记”了承诺或先前的偏好。
    * **浪费信号：** 较早的轮次中可能包含可复用的知识（需求、约束），却被丢弃了。
    * **仍可能出现 token 激增：** 如果最近某一轮包含巨大的工具负载，最近 N 轮仍可能撑爆上下文。

  - **最适用于**

    - 对话中的各项任务彼此独立，上下文互不重叠，不需要把之前的细节带到后面。
    - 你需要可预测性、易于评估和低延迟（运维自动化、CRM/API 操作）。
    - 对话中有用的上下文是局部的（最近的步骤远比久远的历史重要）。

- **上下文摘要** – 将先前的消息（assistant、user、工具等）压缩成结构化的简短摘要，并注入到对话历史中。

  - **优点**

    * **紧凑地保留长程记忆：** 过去的需求、决策和理由在超出 N 轮之后依然存在。
    * **更流畅的用户体验：** 智能体能在长会话中“记住”承诺和约束。
    * **成本可控的扩展：** 一份简洁的摘要可以替代数百轮对话。
    * **可搜索的锚点：** 一条合成的 assistant 消息成为稳定的“迄今为止的世界状态”。

    **缺点**

    * **摘要损失和偏差：** 细节可能被丢弃或权重失衡；细微的约束可能消失。
    * **延迟和成本激增：** 每次刷新都会增加模型工作量（并可能涉及工具裁剪逻辑）。
    * **错误累积：** 如果一个错误事实进入了摘要，它可能会**毒害**后续行为（“上下文中毒”）。
    * **可观测性复杂：** 你必须记录摘要的提示/输出，以便审计和评估。

  - **最适用于**

    - 任务需要在整个流程中收集上下文的用例，例如规划/辅导、重度依赖 RAG 的分析、政策问答。
    - 你需要在长时间跨度内保持连续性，并且必须保留重要细节来解决相关任务。
    - 会话超过 N 轮，但必须可靠地保留决策、ID 和约束。
<br>

**快速对比**

| 维度         | **裁剪（最近 N 轮）**         | **摘要（较早内容 → 生成的摘要）** |
| ----------------- | ------------------------------- | ------------------------------------ |
| 延迟 / 成本    | 最低（无额外调用）     | 在摘要刷新时较高 |
| 长程回忆 | 弱（硬性截断）         | 强（紧凑地向前携带）   |
| 风险类型         | 上下文丢失                | 上下文失真/中毒     |
| 可观测性     | 日志简单                 | 必须记录摘要提示/输出 |
| 评估稳定性    | 高                        | 需要稳健的摘要评估       |
| 最适用于          | 重度使用工具的运维、短工作流 | 分析师/礼宾类场景、长对话      |


## 前提条件

在运行本 cookbook 之前，你必须设置以下账户并完成一些设置操作。这些前提条件对于与本项目中使用的 API 交互至关重要。

#### 第 0 步：OpenAI 账户和 `OPENAI_API_KEY`

- **目的：**  
  你需要一个 OpenAI 账户才能访问语言模型并使用本 cookbook 中介绍的 Agents SDK。

- **操作：**  
  如果你还没有账户，请[注册一个 OpenAI 账户](https://openai.com)。拥有账户后，访问 [OpenAI API Keys 页面](https://platform.openai.com/api-keys) 创建一个 API 密钥。

**在运行工作流之前，请设置环境变量：**

```
# Your openai key
os.environ["OPENAI_API_KEY"] = "sk-proj-..."
```

或者，你也可以导入 agents 库，通过 `set_default_openai_key` 函数设置供智能体使用的 OpenAI API 密钥。

```
from agents import set_default_openai_key
set_default_openai_key("YOUR_API_KEY")
```

#### 第 1 步：安装所需的库

下面我们安装 `openai-agents` 库（[OpenAI Agents SDK](https://github.com/openai/openai-agents-python)）

In [ ]:
%pip install openai-agents nest_asyncio

In [1]:
from openai import OpenAI

client = OpenAI()

In [2]:
from agents import set_tracing_disabled
set_tracing_disabled(True)

我们通过定义并运行一个智能体来测试已安装的库。

In [ ]:
import asyncio
from agents import Agent, Runner


agent = Agent(
    name="Assistant",
    instructions="Reply very concisely.",
)

result = await Runner.run(agent, "Tell me why it is important to evaluate AI agents.")
print(result.final_output)


Evaluating AI agents ensures reliability, safety, ethical alignment, performance accuracy, and helps avoid biases, improving overall trust and effectiveness.


### 定义智能体

首先，我们可以从 Agents SDK 库中定义所需的组件。在创建智能体时根据用例添加指令。

#### 客户服务智能体

In [214]:
support_agent = Agent(
    name="Customer Support Assistant",
    model="gpt-5",
    instructions=(
        "You are a patient, step-by-step IT support assistant. "
        "Your role is to help customers troubleshoot and resolve issues with devices and software. "
        "Guidelines:\n"
        "- Be concise and use numbered steps where possible.\n"
        "- Ask only one focused, clarifying question at a time before suggesting next actions.\n"
        "- Track and remember multiple issues across the conversation; update your understanding as new problems emerge.\n"
        "- When a problem is resolved, briefly confirm closure before moving to the next.\n"
    )
)


## 上下文裁剪

#### 实现自定义 Session 对象

我们使用的是 [OpenAI Agents Python SDK](https://openai.github.io/openai-agents-python/) 中的 [Session](https://openai.github.io/openai-agents-python/sessions/) 对象。下面是一个 `TrimmingSession` 实现，它**只保留最近 N 轮**（一“轮”= 一条用户消息以及直到下一条用户消息之前的所有内容——包括 assistant 回复和任何工具调用/结果）。它存储在内存中，并在每次写入和读取时自动裁剪。


In [252]:
from __future__ import annotations

import asyncio
from collections import deque
from typing import Any, Deque, Dict, List, cast

from agents.memory.session import SessionABC
from agents.items import TResponseInputItem  # dict-like item

ROLE_USER = "user"


def _is_user_msg(item: TResponseInputItem) -> bool:
    """Return True if the item represents a user message."""
    # Common dict-shaped messages
    if isinstance(item, dict):
        role = item.get("role")
        if role is not None:
            return role == ROLE_USER
        # Some SDKs: {"type": "message", "role": "..."}
        if item.get("type") == "message":
            return item.get("role") == ROLE_USER
    # Fallback: objects with a .role attr
    return getattr(item, "role", None) == ROLE_USER


class TrimmingSession(SessionABC):
    """
    Keep only the last N *user turns* in memory.

    A turn = a user message and all subsequent items (assistant/tool calls/results)
    up to (but not including) the next user message.
    """

    def __init__(self, session_id: str, max_turns: int = 8):
        self.session_id = session_id
        self.max_turns = max(1, int(max_turns))
        self._items: Deque[TResponseInputItem] = deque()  # chronological log
        self._lock = asyncio.Lock()

    # ---- SessionABC API ----

    async def get_items(self, limit: int | None = None) -> List[TResponseInputItem]:
        """Return history trimmed to the last N user turns (optionally limited to most-recent `limit` items)."""
        async with self._lock:
            trimmed = self._trim_to_last_turns(list(self._items))
            return trimmed[-limit:] if (limit is not None and limit >= 0) else trimmed

    async def add_items(self, items: List[TResponseInputItem]) -> None:
        """Append new items, then trim to last N user turns."""
        if not items:
            return
        async with self._lock:
            self._items.extend(items)
            trimmed = self._trim_to_last_turns(list(self._items))
            self._items.clear()
            self._items.extend(trimmed)

    async def pop_item(self) -> TResponseInputItem | None:
        """Remove and return the most recent item (post-trim)."""
        async with self._lock:
            return self._items.pop() if self._items else None

    async def clear_session(self) -> None:
        """Remove all items for this session."""
        async with self._lock:
            self._items.clear()

    # ---- Helpers ----

    def _trim_to_last_turns(self, items: List[TResponseInputItem]) -> List[TResponseInputItem]:
        """
        Keep only the suffix containing the last `max_turns` user messages and everything after
        the earliest of those user messages.

        If there are fewer than `max_turns` user messages (or none), keep all items.
        """
        if not items:
            return items

        count = 0
        start_idx = 0  # default: keep all if we never reach max_turns

        # Walk backward; when we hit the Nth user message, mark its index.
        for i in range(len(items) - 1, -1, -1):
            if _is_user_msg(items[i]):
                count += 1
                if count == self.max_turns:
                    start_idx = i
                    break

        return items[start_idx:]

    # ---- Optional convenience API ----

    async def set_max_turns(self, max_turns: int) -> None:
        async with self._lock:
            self.max_turns = max(1, int(max_turns))
            trimmed = self._trim_to_last_turns(list(self._items))
            self._items.clear()
            self._items.extend(trimmed)

    async def raw_items(self) -> List[TResponseInputItem]:
        """Return the untrimmed in-memory log (for debugging)."""
        async with self._lock:
            return list(self._items)


我们来定义已实现的自定义 session 对象，并设置 max_turns=3。

In [280]:
# Keep only the last 8 turns (user + assistant/tool interactions)
session = TrimmingSession("my_session", max_turns=3)

**如何选择合适的 `max_turns`？**

确定这个参数通常需要用你的对话历史进行实验。一种方法是提取所有对话的总轮数并分析其分布。另一种方法是使用 LLM 来评估对话——识别每段对话包含多少个任务或问题，并计算每个问题平均需要的轮数。


In [281]:
message = "There is a red light blinking on my laptop."

In [282]:
result = await Runner.run(
    support_agent,
    message,
    session=session
)

In [283]:
history = await session.get_items()


In [284]:
history

[{'content': 'There is a red light blinking on my laptop.', 'role': 'user'},
 {'id': 'rs_68be66229c008190aa4b3c5501f397080fdfa41323fb39cb',
  'summary': [],
  'type': 'reasoning',
  'content': []},
 {'id': 'msg_68be662f704c8190969bdf539701a3e90fdfa41323fb39cb',
  'content': [{'annotations': [],
    'text': 'A blinking red light usually indicates a power/battery or hardware fault, but the meaning varies by brand.\n\nWhat is the exact make and model of your laptop?\n\nWhile you check that, please try these quick checks:\n1) Note exactly where the red LED is (charging port, power button, keyboard edge) and the blink pattern (e.g., constant blink, 2 short/1 long).\n2) Plug the charger directly into a known‑good wall outlet (no power strip), ensure the charger tip is fully seated, and look for damage to the cable/port. See if the LED behavior changes.\n3) Leave it on charge for 30 minutes in case the battery is critically low.\n4) Power reset: unplug the charger; if the battery is removable

In [285]:
# Example flow
await session.add_items([{"role": "user", "content": "I am using a macbook pro and it has some overheating issues too."}])
await session.add_items([{"role": "assistant", "content": "I see. Let's check your firmware version."}])
await session.add_items([{"role": "user", "content": "Firmware v1.0.3; still failing."}])
await session.add_items([{"role": "assistant", "content": "Could you please try a factory reset?"}])
await session.add_items([{"role": "user", "content": "Reset done; error 42 now."}])
await session.add_items([{"role": "assistant", "content": "Leave it on charge for 30 minutes in case the battery is critically low. Is there any other error message?"}])
await session.add_items([{"role": "user", "content": "Yes, I see error 404 now."}])
await session.add_items([{"role": "assistant", "content": "Do you see it on the browser while accessing a website?"}])
# At this point, with max_turns=3, everything *before* the earliest of the last 3 user
# messages is summarized into a synthetic pair, and the last 3 turns remain verbatim.

history = await session.get_items()
# Pass `history` into your agent runner / responses call as the conversation context.


In [286]:
len(history)

6

In [287]:
history

[{'role': 'user', 'content': 'Firmware v1.0.3; still failing.'},
 {'role': 'assistant', 'content': 'Could you please try a factory reset?'},
 {'role': 'user', 'content': 'Reset done; error 42 now.'},
 {'role': 'assistant',
  'content': 'Leave it on charge for 30 minutes in case the battery is critically low. Is there any other error message?'},
 {'role': 'user', 'content': 'Yes, I see error 404 now.'},
 {'role': 'assistant',
  'content': 'Do you see it on the browser while accessing a website?'}]

下面你可以看到裁剪 session 在 max_turns=3 时的工作方式。

![Session 中的上下文裁剪](../../images/trimingSession.jpg)

**什么算作一“轮”**

* 一**轮** = 一条**用户**消息**以及其后的所有内容**（assistant 回复、推理、工具调用、工具结果），**直到下一条用户消息为止**。

**何时进行裁剪**

* **写入**时：`add_items(...)` 追加新条目，然后立即裁剪存储的历史记录。
* **读取**时：`get_items(...)` 返回**裁剪后**的视图（因此即使你绕过了写入，读取也不会泄露旧的轮次）。

**如何决定保留哪些内容**

1. 将任何 `role == "user"` 的条目视为**用户消息**（通过 `_is_user_msg`）。
2. **从后往前**扫描历史记录，收集最近 **N** 条用户消息的索引（`max_turns`）。
3. 找出这 N 条用户消息中**最早**的索引。
4. **保留从该索引到末尾的所有内容**；丢弃其之前的所有内容。

这样可以保留每一轮的完整边界：如果保留的最早用户消息位于索引 `k`，那么 `k` 之后的所有 assistant/工具条目也都会被保留。

**小示例**

历史记录（旧 → 新）：

```
0: user("Hi")
1: assistant("Hello!")
2: tool_call("lookup")
3: tool_result("…")
4: user("It didn't work")
5: assistant("Try rebooting")
6: user("Rebooted, now error 42")
7: assistant("On it")
```

当 `max_turns = 2` 时，最近两条用户消息位于索引 **4** 和 **6**。
其中最早的是 **4** → 保留条目 **4..7**，丢弃 **0..3**。

**为什么这种方法效果很好**

* 你始终保留**完整**的轮次，因此 assistant 能保留它所需的直接上下文（既包括用户最近的请求，也包括其间的 assistant/工具步骤）。
* 它通过整轮丢弃较早的内容（而不仅仅是单条消息）来防止上下文膨胀。

**可定制的选项**

* 在初始化时修改 `max_turns`。
* 如果你的条目 schema 不同，请调整 `_is_user_msg(...)`。
* 如果你更希望按**消息数量**或 **token** 设置上限，请替换 `_trim_to_last_turns(...)`，或添加第二轮处理来统计 token。


## 上下文摘要

一旦历史记录超过 `max_turns`，它会完整保留最近 N 个用户轮次，并**将更早的所有内容摘要为两条合成消息**：

* `user`：*"Summarize the conversation we had so far."*
* `assistant`：*{generated summary}*

添加这条由用户发出、请求进行摘要的影子提示，是为了保持对话的自然流程，而不会打乱用户与 assistant 之间的聊天流。生成摘要的最终版本会被注入到 assistant 消息中。

**摘要提示**



精心设计的摘要提示对于保留对话上下文至关重要，并且应始终针对具体用例进行定制。可以把它想象成**一名客服人员将案例移交给下一名客服**。对方需要哪些简洁但关键的细节才能顺利接手？提示应当把握好平衡：既不塞满不必要的信息，也不能过于简略以致丢失关键上下文。实现这种平衡需要精心设计和持续实验，以微调细节的程度。

In [66]:
SUMMARY_PROMPT = """
You are a senior customer-support assistant for tech devices, setup, and software issues.
Compress the earlier conversation into a precise, reusable snapshot for future turns.

Before you write (do this silently):
- Contradiction check: compare user claims with system instructions and tool definitions/logs; note any conflicts or reversals.
- Temporal ordering: sort key events by time; the most recent update wins. If timestamps exist, keep them.
- Hallucination control: if any fact is uncertain/not stated, mark it as UNVERIFIED rather than guessing.

Write a structured, factual summary ≤ 200 words using the sections below (use the exact headings):

• Product & Environment:
  - Device/model, OS/app versions, network/context if mentioned.

• Reported Issue:
  - Single-sentence problem statement (latest state).

• Steps Tried & Results:
  - Chronological bullets (include tool calls + outcomes, errors, codes).

• Identifiers:
  - Ticket #, device serial/model, account/email (only if provided).

• Timeline Milestones:
  - Key events with timestamps or relative order (e.g., 10:32 install → 10:41 error).

• Tool Performance Insights:
  - What tool calls worked/failed and why (if evident).

• Current Status & Blockers:
  - What’s resolved vs pending; explicit blockers preventing progress.

• Next Recommended Step:
  - One concrete action (or two alternatives) aligned with policies/tools.

Rules:
- Be concise, no fluff; use short bullets, verbs first.
- Do not invent new facts; quote error strings/codes exactly when available.
- If previous info was superseded, note “Superseded:” and omit details unless critical.
"""


**设计记忆摘要提示的关键原则**

* **里程碑：** 突出对话中的重要事件——例如问题何时得到解决、何时发现了有价值的信息，或何时收集齐了所有必要的细节。

* **针对具体用例：** 根据具体用例定制压缩提示。想一想人类在解决同样的任务时，会如何在工作记忆中跟踪和回忆信息。

* **矛盾检查：** 确保摘要不会与自身、系统指令或工具定义相冲突。这对推理模型尤其关键，因为它们更容易受到上下文中冲突的影响。

* **时间戳与时间顺序：** 在摘要中纳入事件发生的时间。这有助于模型按顺序推理更新，并减少在时间线上遗忘或记起最新记忆时的混淆。

* **分块：** 将细节组织成类别或章节，而不是冗长的段落。结构化的分组能提升 LLM 理解信息之间关系的能力。

* **工具表现洞察：** 记录从多轮、使用工具的交互中学到的经验——例如，记下哪些工具对特定查询效果好以及原因。这些洞察对于指导后续步骤很有价值。

* **指导与示例：** 用清晰的指导来引导摘要。在可能的情况下，从对话历史中提取具体示例，使后续轮次更有依据、上下文更丰富。

* **幻觉控制：** 对纳入的内容要精确。摘要中哪怕是轻微的幻觉也可能向后传播，用不准确的信息污染后续的上下文。

* **模型选择：** 根据用例需求、摘要长度以及延迟与成本之间的权衡来选择摘要模型。在某些情况下，使用与 AI 智能体本身相同的模型可能更有优势。


In [289]:
class LLMSummarizer:
    def __init__(self, client, model="gpt-4o", max_tokens=400, tool_trim_limit=600):
        self.client = client
        self.model = model
        self.max_tokens = max_tokens
        self.tool_trim_limit = tool_trim_limit

    async def summarize(self, messages: List[Item]) -> Tuple[str, str]:
        """
        Create a compact summary from `messages`.

        Returns:
            Tuple[str, str]: The shadow user line to keep dialog natural,
            and the model-generated summary text.
        """
        user_shadow = "Summarize the conversation we had so far."
        TOOL_ROLES = {"tool", "tool_result"}

        def to_snippet(m: Item) -> str | None:
            role = (m.get("role") or "assistant").lower()
            content = (m.get("content") or "").strip()
            if not content:
                return None
            # Trim verbose tool outputs to keep prompt compact    
            if role in TOOL_ROLES and len(content) > self.tool_trim_limit:
                content = content[: self.tool_trim_limit] + " …"
            return f"{role.upper()}: {content}"

        # Build compact, trimmed history
        history_snippets = [s for m in messages if (s := to_snippet(m))]

        prompt_messages = [
            {"role": "system", "content": SUMMARY_PROMPT},
            {"role": "user", "content": "\n".join(history_snippets)},
        ]

        resp = await asyncio.to_thread(
            self.client.responses.create,
            model=self.model,
            input=prompt_messages,
            max_output_tokens=self.max_tokens,
        )

        summary = resp.output_text
        await asyncio.sleep(0)  # yield control
        return user_shadow, summary


In [296]:
import asyncio
from collections import deque
from typing import Optional, List, Tuple, Dict, Any

Record = Dict[str, Dict[str, Any]]  # {"msg": {...}, "meta": {...}}

class SummarizingSession:
    """
    Session that keeps only the last N *user turns* verbatim and summarizes the rest.

    - A *turn* starts at a real user message and includes everything until the next real user message.
    - When the number of real user turns exceeds `context_limit`, everything before the earliest
      of the last `keep_last_n_turns` user-turn starts is summarized into a synthetic user→assistant pair.
    - Stores full records (message + metadata). Exposes:
        • get_items():           model-safe messages only (no metadata)
        • get_full_history():    [{"message": msg, "metadata": meta}, ...]
    """

    # Only these keys are ever sent to the model; the rest live in metadata.
    _ALLOWED_MSG_KEYS = {"role", "content", "name"}

    def __init__(
        self,
        keep_last_n_turns: int = 3,
        context_limit: int = 3,
        summarizer: Optional["Summarizer"] = None,
        session_id: Optional[str] = None,
    ):
        assert context_limit >= 1
        assert keep_last_n_turns >= 0
        assert keep_last_n_turns <= context_limit, "keep_last_n_turns should not be greater than context_limit"

        self.keep_last_n_turns = keep_last_n_turns
        self.context_limit = context_limit
        self.summarizer = summarizer
        self.session_id = session_id or "default"

        self._records: deque[Record] = deque()
        self._lock = asyncio.Lock()

    # --------- public API used by your runner ---------
    async def get_items(self, limit: Optional[int] = None) -> List[Dict[str, Any]]:
        """Return model-safe messages only (no metadata)."""
        async with self._lock:
            data = list(self._records)
        msgs = [self._sanitize_for_model(rec["msg"]) for rec in data]
        return msgs[-limit:] if limit else msgs

    async def add_items(self, items: List[Dict[str, Any]]) -> None:
        """Append new items and, if needed, summarize older turns."""
        # 1) Ingest items
        async with self._lock:
            for it in items:
                msg, meta = self._split_msg_and_meta(it)
                self._records.append({"msg": msg, "meta": meta})

            need_summary, boundary = self._summarize_decision_locked()

        # 2) No summarization needed → just normalize flags and exit
        if not need_summary:
            async with self._lock:
                self._normalize_synthetic_flags_locked()
            return

        # 3) Prepare summary prefix (model-safe copy) outside the lock
        async with self._lock:
            snapshot = list(self._records)
            prefix_msgs = [r["msg"] for r in snapshot[:boundary]]

        user_shadow, assistant_summary = await self._summarize(prefix_msgs)

        # 4) Re-check and apply summary atomically
        async with self._lock:
            still_need, new_boundary = self._summarize_decision_locked()
            if not still_need:
                self._normalize_synthetic_flags_locked()
                return

            snapshot = list(self._records)
            suffix = snapshot[new_boundary:]  # keep-last-N turns live here

            # Replace with: synthetic pair + suffix
            self._records.clear()
            self._records.extend([
                {
                    "msg": {"role": "user", "content": user_shadow},
                    "meta": {
                        "synthetic": True,
                        "kind": "history_summary_prompt",
                        "summary_for_turns": f"< all before idx {new_boundary} >",
                    },
                },
                {
                    "msg": {"role": "assistant", "content": assistant_summary},
                    "meta": {
                        "synthetic": True,
                        "kind": "history_summary",
                        "summary_for_turns": f"< all before idx {new_boundary} >",
                    },
                },
            ])
            self._records.extend(suffix)

            # Ensure all real user/assistant messages explicitly have synthetic=False
            self._normalize_synthetic_flags_locked()

    async def pop_item(self) -> Optional[Dict[str, Any]]:
        """Pop the latest message (model-safe), if any."""
        async with self._lock:
            if not self._records:
                return None
            rec = self._records.pop()
            return dict(rec["msg"])

    async def clear_session(self) -> None:
        """Remove all records."""
        async with self._lock:
            self._records.clear()

    def set_max_turns(self, n: int) -> None:
        """
        Back-compat shim for old callers: update `context_limit`
        and clamp `keep_last_n_turns` if needed.
        """
        assert n >= 1
        self.context_limit = n
        if self.keep_last_n_turns > self.context_limit:
            self.keep_last_n_turns = self.context_limit

    # Full history (debugging/analytics/observability)

    async def get_full_history(self, limit: Optional[int] = None) -> List[Dict[str, Any]]:
        """
        Return combined history entries in the shape:
          {"message": {role, content[, name]}, "metadata": {...}}
        This is NOT sent to the model; for logs/UI/debugging only.
        """
        async with self._lock:
            data = list(self._records)
        out = [{"message": dict(rec["msg"]), "metadata": dict(rec["meta"])} for rec in data]
        return out[-limit:] if limit else out

    # Back-compat alias
    async def get_items_with_metadata(self, limit: Optional[int] = None) -> List[Dict[str, Any]]:
        return await self.get_full_history(limit)

    # Internals

    def _split_msg_and_meta(self, it: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """
        Split input into (msg, meta):
          - msg keeps only _ALLOWED_MSG_KEYS; if role/content missing, default them.
          - everything else goes under meta (including nested "metadata" if provided).
          - default synthetic=False for real user/assistant unless explicitly set.
        """
        msg = {k: v for k, v in it.items() if k in self._ALLOWED_MSG_KEYS}
        extra = {k: v for k, v in it.items() if k not in self._ALLOWED_MSG_KEYS}
        meta = dict(extra.pop("metadata", {}))
        meta.update(extra)

        msg.setdefault("role", "user")
        msg.setdefault("content", str(it))

        role = msg.get("role")
        if role in ("user", "assistant") and "synthetic" not in meta:
            meta["synthetic"] = False
        return msg, meta

    @staticmethod
    def _sanitize_for_model(msg: Dict[str, Any]) -> Dict[str, Any]:
        """Drop anything not allowed in model calls."""
        return {k: v for k, v in msg.items() if k in SummarizingSession._ALLOWED_MSG_KEYS}

    @staticmethod
    def _is_real_user_turn_start(rec: Record) -> bool:
        """True if record starts a *real* user turn (role=='user' and not synthetic)."""
        return (
            rec["msg"].get("role") == "user"
            and not rec["meta"].get("synthetic", False)
        )

    def _summarize_decision_locked(self) -> Tuple[bool, int]:
        """
        Decide whether to summarize and compute the boundary index.

        Returns:
            (need_summary, boundary_idx)

        If need_summary:
          • boundary_idx is the earliest index among the last `keep_last_n_turns`
            *real* user-turn starts.
          • Everything before boundary_idx becomes the summary prefix.
        """
        user_starts: List[int] = [
            i for i, rec in enumerate(self._records) if self._is_real_user_turn_start(rec)
        ]
        real_turns = len(user_starts)

        # Not over the limit → nothing to do
        if real_turns <= self.context_limit:
            return False, -1

        # Keep zero turns verbatim → summarize everything
        if self.keep_last_n_turns == 0:
            return True, len(self._records)

        # Otherwise, keep the last N turns; summarize everything before the earliest of those
        if len(user_starts) < self.keep_last_n_turns:
            return False, -1  # defensive (shouldn't happen given the earlier check)

        boundary = user_starts[-self.keep_last_n_turns]

        # If there is nothing before boundary, there is nothing to summarize
        if boundary <= 0:
            return False, -1

        return True, boundary

    def _normalize_synthetic_flags_locked(self) -> None:
        """Ensure all real user/assistant records explicitly carry synthetic=False."""
        for rec in self._records:
            role = rec["msg"].get("role")
            if role in ("user", "assistant") and "synthetic" not in rec["meta"]:
                rec["meta"]["synthetic"] = False

    async def _summarize(self, prefix_msgs: List[Dict[str, Any]]) -> Tuple[str, str]:
        """
        Ask the configured summarizer to compress the given prefix.
        Uses model-safe messages only. If no summarizer is configured,
        returns a graceful fallback.
        """
        if not self.summarizer:
            return ("Summarize the conversation we had so far.", "Summary unavailable.")
        clean_prefix = [self._sanitize_for_model(m) for m in prefix_msgs]
        return await self.summarizer.summarize(clean_prefix)


![Session 中的上下文裁剪](../../images/summarizingSession.jpg)

**总体思路**

* 一**轮** = 一条**真实用户**消息**以及其后的所有内容**（assistant 回复、工具调用/结果等），**直到下一条真实用户消息为止**。
* 你需要配置两个参数：

  * **`context_limit`**：在进行摘要之前，原始历史记录中允许的**真实用户轮次**的最大数量。
  * **`keep_last_n_turns`**：进行摘要时，按原样保留最近多少**轮**。

    * 不变式：`keep_last_n_turns <= context_limit`。
* 当**真实**用户轮次的数量超过 `context_limit` 时，session 会：

  1. **摘要**最近 `keep_last_n_turns` 轮中最早一轮开始**之前**的所有内容，
  2. 在保留区域的顶部注入一对**合成的 user→assistant 消息**：

     * `user`：`"Summarize the conversation we had so far."`（影子提示）
     * `assistant`：`{generated summary}`
  3. **按原样保留**最近 `keep_last_n_turns` 轮。

这保证了最近 `keep_last_n_turns` 轮会完全按原样保留，而所有更早的内容都被压缩成这两条合成消息。


In [291]:
session = SummarizingSession(
    keep_last_n_turns=2,
    context_limit=4,
    summarizer=LLMSummarizer(client)
)

In [292]:

# Example flow
await session.add_items([{"role": "user", "content": "Hi, my router won't connect. by the way, I am using Windows 10. I tried troubleshooting via your FAQs but I didn't get anywhere. This is my third tiem calling you. I am based in the US and one of Premium customers."}])
await session.add_items([{"role": "assistant", "content": "Let's check your firmware version."}])
await session.add_items([{"role": "user", "content": "Firmware v1.0.3; still failing."}])
await session.add_items([{"role": "assistant", "content": "Try a factory reset."}])
await session.add_items([{"role": "user", "content": "Reset done; error 42 now."}])
await session.add_items([{"role": "assistant", "content": "Try to install a new firmware."}])
await session.add_items([{"role": "user", "content": "I tried but I got another error now."}])
await session.add_items([{"role": "assistant", "content": "Can you please provide me with the error code?"}])
await session.add_items([{"role": "user", "content": "It says 404 not found when I try to access the page."}])
await session.add_items([{"role": "assistant", "content": "Are you connected to the internet?"}])
# At this point, with context_limit=4, everything *before* the earliest of the last 4 turns
# is summarized into a synthetic pair, and the last 2 turns remain verbatim.


In [293]:
history = await session.get_items()
# Pass `history` into your agent runner / responses call as the conversation context.

In [294]:
history

[{'role': 'user', 'content': 'Summarize the conversation we had so far.'},
 {'role': 'assistant',
  'content': '• Product & Environment:\n  - Router with Firmware v1.0.3, Windows 10, based in the US.\n\n• Reported Issue:\n  - Router fails to connect.\n\n• Steps Tried & Results:\n  - Checked FAQs: No resolution.\n  - Checked firmware version: v1.0.3, problem persists.\n  - Factory reset: Resulted in error 42.\n\n• Identifiers:\n  - Premium customer (no specific identifier provided).\n\n• Timeline Milestones:\n  - Initial troubleshooting via FAQs.\n  - Firmware check (before factory reset).\n  - Factory reset → Error 42.\n\n• Tool Performance Insights:\n  - Firmware version check successful.\n  - Factory reset resulted in new error (42).\n\n• Current Status & Blockers:\n  - Connection issue unresolved; error 42 is the immediate blocker.\n\n• Next Recommended Step:\n  - Install a new firmware update.'},
 {'role': 'user', 'content': 'I tried but I got another error now.'},
 {'role': 'assis

In [295]:
print(history[1]['content'])

• Product & Environment:
  - Router with Firmware v1.0.3, Windows 10, based in the US.

• Reported Issue:
  - Router fails to connect.

• Steps Tried & Results:
  - Checked FAQs: No resolution.
  - Checked firmware version: v1.0.3, problem persists.
  - Factory reset: Resulted in error 42.

• Identifiers:
  - Premium customer (no specific identifier provided).

• Timeline Milestones:
  - Initial troubleshooting via FAQs.
  - Firmware check (before factory reset).
  - Factory reset → Error 42.

• Tool Performance Insights:
  - Firmware version check successful.
  - Factory reset resulted in new error (42).

• Current Status & Blockers:
  - Connection issue unresolved; error 42 is the immediate blocker.

• Next Recommended Step:
  - Install a new firmware update.


你可以使用 get_items_with_metadata 方法获取 session 的完整历史记录（包括元数据），以便进行调试和分析。

In [ ]:
full_history = await session.get_items_with_metadata()

In [209]:
full_history

[{'message': {'role': 'user',
   'content': 'Summarize the conversation we had so far.'},
  'metadata': {'synthetic': True,
   'kind': 'history_summary_prompt',
   'summary_for_turns': '< all before idx 6 >'}},
 {'message': {'role': 'assistant',
   'content': '**Product & Environment:**\n- Device: Router\n- OS: Windows 10\n- Firmware: v1.0.3\n\n**Reported Issue:**\n- Router fails to connect to the internet, now showing error 42.\n\n**Steps Tried & Results:**\n- Checked FAQs: No resolution.\n- Firmware version checked: v1.0.3.\n- Factory reset performed: Resulted in error 42.\n\n**Identifiers:**\n- UNVERIFIED\n\n**Timeline Milestones:**\n- User attempted FAQ troubleshooting.\n- Firmware checked after initial advice.\n- Factory reset led to error 42.\n\n**Tool Performance Insights:**\n- FAQs and basic reset process did not resolve the issue.\n\n**Current Status & Blockers:**\n- Error 42 unresolved; firmware update needed.\n\n**Next Recommended Step:**\n- Install the latest firmware updat

In [210]:
print(history[1]['content'])

**Product & Environment:**
- Device: Router
- OS: Windows 10
- Firmware: v1.0.3

**Reported Issue:**
- Router fails to connect to the internet, now showing error 42.

**Steps Tried & Results:**
- Checked FAQs: No resolution.
- Firmware version checked: v1.0.3.
- Factory reset performed: Resulted in error 42.

**Identifiers:**
- UNVERIFIED

**Timeline Milestones:**
- User attempted FAQ troubleshooting.
- Firmware checked after initial advice.
- Factory reset led to error 42.

**Tool Performance Insights:**
- FAQs and basic reset process did not resolve the issue.

**Current Status & Blockers:**
- Error 42 unresolved; firmware update needed.

**Next Recommended Step:**
- Install the latest firmware update and check for resolution.


### 说明与设计选择

* **在“新鲜”一侧保留轮次边界**：**`keep_last_n_turns` 个用户轮次**保持原样；更早的内容都被压缩。
* **双消息摘要块**：便于下游工具检测或显示（`metadata.synthetic == True`）。
* **异步 + 锁规范**：在（可能较慢的）摘要运行期间我们会**释放锁**；然后在应用摘要之前重新检查条件，以避免出现竞争性的合并。
* **幂等行为**：如果在摘要期间有更多消息到达，await 之后的重新检查可以防止用过时的内容进行改写。


## 评估

归根结底，对于上下文工程来说，**你需要的也只是评估（evals is all you need）**。需要问的关键问题是：*我们如何知道模型没有“丢失上下文”或“混淆上下文”？*

虽然将来可以围绕记忆单独写一份完整的 cookbook，但这里先提供一些轻量级评估框架的思路：

* **基线与增量：** 继续运行你的核心评估集，并比较实验前后的结果，以衡量记忆方面的改进。
* **LLM-as-Judge：** 使用一个配有精心设计的评分提示的模型来评估摘要质量。重点关注它是否以正确的格式捕获了最重要的细节。
* **对话记录重放：** 重新运行长对话，并在启用和不启用上下文裁剪的情况下测量下一轮的准确率。指标可以包括实体/ID 的精确匹配，以及基于评分细则的推理质量评分。
* **错误回归跟踪：** 留意常见的失败模式——未回答的问题、丢失的约束，或不必要/重复的工具调用。
* **token 压力检查：** 标记 token 限制迫使丢弃受保护上下文的情况。记录前后的 token 数量，以发现关键细节何时被删减。

---